# Plot H_res Ablation Curves

This notebook parses a `hres_ablations.log` file produced by `run_hres_ablations.sh` and plots loss curves for the three sequential runs:

- `mhc`
- `mhc_identity`
- `mhc_orthogonal`

It is designed for offline analysis on a different server. Point `LOG_PATH` at the log file on that machine and run the notebook top to bottom.

In [ ]:
from pathlib import Path
import math
import re
import statistics

import matplotlib.pyplot as plt

LOG_PATH = Path("hres_ablations.log")
RUN_LABELS = ["mhc", "mhc_identity", "mhc_orthogonal"]

# Filters for logs that contain retries, restarts, or manually appended reruns.
MIN_LAST_ITER = 2500
TARGET_LAST_ITER = 3000
SELECT_RUN_INDICES = None  # e.g. [0, 1, 3] after inspecting the complete-run table below
AUTO_SELECT_LAST_COMPLETE = False

plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.grid"] = True

assert LOG_PATH.exists(), f"Log file not found: {LOG_PATH.resolve()}"


In [ ]:
TRAIN_RE = re.compile(
    r"^iter\s+(?P<iter>\d+):\s+loss\s+(?P<loss>[-+0-9.eE]+),\s+lr\s+(?P<lr>[-+0-9.eE]+),\s+time\s+(?P<time_ms>[-+0-9.eE]+)ms,\s+tok/s\s+(?P<tok_s>[-+0-9.eE]+)"
)
EVAL_RE = re.compile(
    r"^iter\s+(?P<iter>\d+):\s+train\s+loss\s+(?P<train_loss>[-+0-9.eE]+),\s+val\s+loss\s+(?P<val_loss>[-+0-9.eE]+)"
)
TOKENS_RE = re.compile(r"^\s*tokens per iteration:\s*(?P<value>[0-9,]+)")
WORLD_RE = re.compile(r"^\s*world_size=(?P<world>\d+),\s*grad_accum_steps=(?P<gas>\d+)")
PARAMS_RE = re.compile(r"^\s*model params:\s*(?P<value>[0-9,]+)")
DATA_RE = re.compile(r"^Train tokens:\s*(?P<train>[0-9,]+),\s*Val tokens:\s*(?P<val>[0-9,]+)")


def _to_number(text):
    value = float(text)
    return int(value) if value.is_integer() else value


def parse_hres_log(path):
    runs = []
    current = None

    for raw_line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip("\n")

        if line.startswith("Training on "):
            if current is not None:
                runs.append(current)
            current = {
                "meta": {"training_header": line},
                "train_records": [],
                "eval_records": [],
            }

        if current is None:
            continue

        if match := TOKENS_RE.match(line):
            current["meta"]["tokens_per_iteration"] = int(match.group("value").replace(",", ""))
            continue

        if match := WORLD_RE.match(line):
            current["meta"]["world_size"] = int(match.group("world"))
            current["meta"]["grad_accum_steps_per_rank"] = int(match.group("gas"))
            continue

        if match := PARAMS_RE.match(line):
            current["meta"]["model_params"] = int(match.group("value").replace(",", ""))
            continue

        if match := DATA_RE.match(line):
            current["meta"]["train_tokens"] = int(match.group("train").replace(",", ""))
            current["meta"]["val_tokens"] = int(match.group("val").replace(",", ""))
            continue

        if match := EVAL_RE.match(line):
            current["eval_records"].append(
                {
                    "iter": int(match.group("iter")),
                    "train_loss": float(match.group("train_loss")),
                    "val_loss": float(match.group("val_loss")),
                }
            )
            continue

        if match := TRAIN_RE.match(line):
            current["train_records"].append(
                {
                    "iter": int(match.group("iter")),
                    "loss": float(match.group("loss")),
                    "lr": float(match.group("lr")),
                    "time_ms": _to_number(match.group("time_ms")),
                    "tok_s": _to_number(match.group("tok_s")),
                }
            )

    if current is not None:
        runs.append(current)

    runs = [run for run in runs if run["train_records"] or run["eval_records"]]

    for idx, run in enumerate(runs):
        run["raw_index"] = idx

    return runs


def summarize_run(run):
    train_records = run["train_records"]
    eval_records = run["eval_records"]

    best_val = min((row["val_loss"] for row in eval_records), default=math.nan)
    best_val_iter = next((row["iter"] for row in eval_records if row["val_loss"] == best_val), None)
    last_eval = eval_records[-1] if eval_records else None
    last_train = train_records[-1] if train_records else None

    return {
        "raw_index": run["raw_index"],
        "label": run.get("label", f"run_{run['raw_index'] + 1}"),
        "last_iter": last_train["iter"] if last_train else None,
        "eval_points": len(eval_records),
        "best_val_loss": best_val,
        "best_val_iter": best_val_iter,
        "last_val_loss": last_eval["val_loss"] if last_eval else math.nan,
        "last_eval_train_loss": last_eval["train_loss"] if last_eval else math.nan,
        "last_train_loss": last_train["loss"] if last_train else math.nan,
        "median_tok_s": statistics.median(row["tok_s"] for row in train_records) if train_records else math.nan,
        "tokens_per_iteration": run["meta"].get("tokens_per_iteration"),
        "world_size": run["meta"].get("world_size"),
        "grad_accum_steps_per_rank": run["meta"].get("grad_accum_steps_per_rank"),
    }


def print_run_table(title, summaries):
    print(title)
    for summary in summaries:
        print(
            f"  idx={summary['raw_index']:>2} "
            f"last_iter={summary['last_iter']} "
            f"best_val={summary['best_val_loss']:.4f} "
            f"last_val={summary['last_val_loss']:.4f} "
            f"median_tok_s={summary['median_tok_s']:.0f} "
            f"eval_points={summary['eval_points']}"
        )


def is_complete_run(run):
    summary = summarize_run(run)
    last_iter = summary["last_iter"]
    if last_iter is None:
        return False
    if TARGET_LAST_ITER is not None:
        return last_iter == TARGET_LAST_ITER
    return last_iter >= MIN_LAST_ITER


def select_runs(complete_runs):
    if SELECT_RUN_INDICES is not None:
        return [complete_runs[idx] for idx in SELECT_RUN_INDICES]

    required = len(RUN_LABELS)
    if len(complete_runs) < required:
        raise AssertionError(
            f"Need at least {required} complete runs, found {len(complete_runs)}. "
            "Lower MIN_LAST_ITER / TARGET_LAST_ITER or set SELECT_RUN_INDICES manually."
        )

    if AUTO_SELECT_LAST_COMPLETE:
        return complete_runs[-required:]
    return complete_runs[:required]


all_runs = parse_hres_log(LOG_PATH)
assert all_runs, "No training runs found in the log."

all_summaries = [summarize_run(run) for run in all_runs]
complete_runs = [run for run in all_runs if is_complete_run(run)]
complete_summaries = [summarize_run(run) for run in complete_runs]
selected_runs = select_runs(complete_runs)

for label, run in zip(RUN_LABELS, selected_runs):
    run["label"] = label

runs = selected_runs
selected_summaries = [summarize_run(run) for run in runs]

print(f"Parsed {len(all_runs)} runs from {LOG_PATH}")
print_run_table("All runs:", all_summaries)
print()
print_run_table("Complete runs:", complete_summaries)
print()
print("Selected runs:")
for summary in selected_summaries:
    print(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for run in runs:
    train_records = run["train_records"]
    eval_records = run["eval_records"]

    if train_records:
        axes[0].plot(
            [row["iter"] for row in train_records],
            [row["loss"] for row in train_records],
            label=run["label"],
            linewidth=1.8,
        )

    if eval_records:
        axes[1].plot(
            [row["iter"] for row in eval_records],
            [row["val_loss"] for row in eval_records],
            marker="o",
            label=run["label"],
            linewidth=1.8,
        )

axes[0].set_title("Train Loss")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")

axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Loss")

for ax in axes:
    ax.legend()

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for run in runs:
    train_records = run["train_records"]
    eval_records = run["eval_records"]

    if train_records:
        axes[0].plot(
            [row["iter"] for row in train_records],
            [row["tok_s"] for row in train_records],
            label=run["label"],
            linewidth=1.8,
        )

    if eval_records:
        axes[1].plot(
            [row["iter"] for row in eval_records],
            [row["train_loss"] for row in eval_records],
            marker="o",
            label=run["label"],
            linewidth=1.8,
        )

axes[0].set_title("Tokens Per Second")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("tok/s")

axes[1].set_title("Eval Train Loss")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Loss")

for ax in axes:
    ax.legend()

plt.show()

In [ ]:
summaries = [summarize_run(run) for run in runs]
for summary in summaries:
    print(
        f"{summary['label']}: "
        f"best_val={summary['best_val_loss']:.4f} @ iter {summary['best_val_iter']}, "
        f"last_val={summary['last_val_loss']:.4f}, "
        f"last_train={summary['last_train_loss']:.4f}, "
        f"median_tok_s={summary['median_tok_s']:.0f}"
    )